## Imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import datetime
import einops
from torch import nn
import torch as t
from typing import List
from jaxtyping import Float, Int
from tyche.inductive_bias import *

import itertools
import plotly.graph_objs as go  # Import the graph objects from Plotly
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
from tqdm.notebook import tqdm
import pandas as pd
from tyche.estimator import VolumeConfig, VolumeEstimator

/home/louis/tyche/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
device = t.device("cuda:7" if t.cuda.is_available() else "cpu")
t.set_default_device(device)

In [4]:
import os
import torch as t
import einops
import plotly.graph_objects as go


def plot_indicator_table(model, params, save=False):
    device = next(model.parameters()).device
    N = params.N
    group_set = [[i, j, (i + j) % N] for i in range(N) for j in range(N)]
    inputs = t.tensor([g[:2] for g in group_set], dtype=t.long).to(device)

    with t.no_grad():
        model.eval()
        logits = model(inputs)  # shape N^2 x N
        max_prob_entry = t.argmax(logits, dim=-1)  # shape N^2

    output_matrix = einops.rearrange(max_prob_entry, "(n m) -> n m", n=N)  # shape N x N
    hover_labels = [[f"{output_matrix[j][i]}" for i in range(N)] for j in range(N)]
    row_labels = [str(g) for g in range(N)]
    col_labels = row_labels

    # Generate N different colors for the heatmap
    import plotly.colors as pc

    # Using a colorscale that works well for categorical data
    if N <= 10:
        # For small N, use distinct colors from the Plotly qualitative colorscales
        colors = pc.qualitative.Plotly[:N]
    else:
        # For larger N, generate a continuous colorscale with N distinct colors
        colorscale = pc.sequential.Viridis
        colors = [pc.sample_colorscale(colorscale, i / (N - 1))[0] for i in range(N)]

    # Create the colorscale with proper scaling between 0 and 1
    custom_colorscale = []
    for i in range(N):
        # Lower bound for this color
        custom_colorscale.append([i / N, colors[i]])
        # Upper bound for this color (except for the last color)
        if i < N - 1:
            custom_colorscale.append([(i + 1) / N, colors[i]])

    fig = go.Figure(
        data=go.Heatmap(
            z=output_matrix.tolist(),
            showscale=False,
            colorscale=custom_colorscale,
            x=col_labels,
            y=row_labels,
            zmin=0,
            zmax=N - 1,
            customdata=hover_labels,
            hovertemplate="x=%{x}<br>"
            + "y=%{y}<br>"
            + "z=%{customdata}<extra></extra>",
        ),
    )

    fig.update_layout(
        title=f"Final run",
        xaxis={
            "showgrid": True,
            "side": "top",
            "ticks": "outside",
            "tickmode": "array",
            "tickvals": [i for i in range(N)],
            "ticktext": row_labels,
        },
        yaxis={
            "showgrid": True,
            # "autorange": "reversed",
            "side": "left",
            "ticks": "outside",
            "tickmode": "array",
            "tickvals": [i for i in range(N)],
            "ticktext": col_labels,
        },
        height=900,
        width=900,
    )

    if save:
        # Create plots directory if it doesn't exist
        if not os.path.exists("plots"):
            os.mkdir("plots")
        fig.write_html("./plots/plot_final.html")

    return fig

## Playground

In [28]:
params = MLPConfig(
    N=53,
    num_additional_layers=0,
    dimensions=(32),
    bias_unembed=True,
    activation=t.nn.ReLU(),
    embed_dimension=16,
    linear_dimension=32,
    W_amplitude=1,
    device="cuda:7",
)
model = MLP_VARIANTS(params)
model.initialize_weights()

In [9]:
MLPConfig(**{})

MLPConfig(activation=ReLU(), N=113, embed_dimension=24, linear_dimension=48, intermediate='pure', embedding_tied=False, unembedding_tied=False, bias_unembed=False, num_additional_layers=0, dimensions=(), bias_layer=False, W_amplitude=1.0, weight_mode='uniform', device='cpu')

In [6]:
test_inputs = t.tensor(
    list(itertools.product(range(params.N), repeat=2)), device=device
)

In [6]:
cfg = VolumeConfig(
    model_type="mlp",
    model_name={"device": "cuda:7"},
    n_samples=100,  # number of MC samples
    iters=15,
    cutoff=1e-2,  # KL-divergence cutoff (nats)
    cache_mode=None,  # see below
    chunking=False,  # whether to use chunk_and_tokenize
    reduction=None,
    device="cuda:7",
    tol=0.035,
)

estimator = VolumeEstimator.from_config(cfg)

In [7]:
k = estimator.run()

  0%|          | 0/100 [00:00<?, ?it/s]

100%|██████████| 100/100 [00:01<00:00, 97.20it/s]


In [20]:
samples = k.estimates.squeeze()

In [13]:
k.estimates.mean()
k.estimates.std()

tensor(125.8676, device='cuda:7')

In [16]:
k.estimates.std() / k.estimates.mean()

tensor(-0.0192, device='cuda:7')

In [19]:
from tyche.convnext import load_convnext_checkpoint


path = "/mnt/ssd-1/adam/basin-volume/runs/b16pai_p001/checkpoint-65536"

model = load_convnext_checkpoint(checkpoint_path=path, device=device)

/mnt/ssd-1/adam/basin-volume/runs/b16pai_p001/checkpoint-65536


In [21]:
from tyche.convnext import load_cifar10_splits


splits = load_cifar10_splits(size=1024)

Map: 100%|██████████| 1024/1024 [00:00<00:00, 2691.97 examples/s]


In [30]:
val_data = splits["val"].to(device=device)

In [31]:
val_data.device

device(type='cuda', index=7)

In [ ]:
model.to(device)

In [36]:
model = model.to(t.float32)

In [39]:
model = model.float()

# Make sure your validation data is also float32 and on the same device
val_data = val_data.to(dtype=t.float32, device=device)

# Now try running the model
outputs = model(val_data)

In [12]:
cfg = VolumeConfig(
    model_type="convnext",
    n_samples=10,  # number of MC samples
    cutoff=1e-2,  # KL-divergence cutoff (nats)
    cache_mode=None,  # see below
    chunking=False,  # whether to use chunk_and_tokenize
    reduction="None",
)

estimator = VolumeEstimator.from_config(cfg)

Map: 100%|██████████| 1024/1024 [00:00<00:00, 2589.43 examples/s]


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cuda:7! (when checking argument for argument weight in method wrapper_CUDA__cudnn_convolution)

In [20]:
# read /home/louis/shared_database.parquet
import pandas as pd
import os
from tqdm import tqdm

# Load the parquet file
df = pd.read_parquet("/home/louis/shared_database.parquet")

In [22]:
df

,activation,depth,weight_scale,sample_id,Volume estimates,N,embed_dimension,linear_dimension,intermediate,embedding_tied,unembedding_tied,bias_unembed,num_additional_layers,dimensions,bias_layer,W_amplitude,weight_mode,device,volume_estimates
0,ReLU,1.0,0.316228,0,"[-16582.102, -15779.354, -16187.324, -15762.04...",NaN,NaN,NaN,None,None,None,None,NaN,NaN,None,NaN,None,None,None
1,ReLU,NaN,NaN,0,None,113.0,24.0,48.0,pure,False,False,False,1.0,48.0,False,0.316228,uniform,cuda:7,"[-16592.807, -16180.58, -16222.717, -15787.801..."


In [13]:
import math
import numpy as np


WEIGHTSCALE = [math.sqrt(10) ** i for i in np.arange(-1, 2, 0.5)]

In [14]:
WEIGHTSCALE

[np.float64(0.31622776601683794),
 np.float64(0.5623413251903491),
 np.float64(1.0),
 np.float64(1.7782794100389228),
 np.float64(3.1622776601683795),
 np.float64(5.623413251903491)]